In [ ]:
import pandas as pd
# Q1. Build Personalized Knowledge Base
# Roll Number: 1024160013
# Last two digits: 1, 3


roll_number = "1024160017"

last_two_digits = [int(d) for d in roll_number[-2:]]

categories = ["billing", "account", "general"]

# Fixed entries given in the assignment
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Personalized entries
personalized_entries = []

for digit in last_two_digits:

    category = categories[digit % 3]

    if category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Go to Settings > Account > Mobile Number to update it.",
            "keywords": "mobile number update",
            "category": category
        }

    elif category == "billing":
        entry = {
            "question": "how can i get a payment receipt",
            "answer": "You can download your payment receipt from the Billing section.",
            "keywords": "receipt payment billing",
            "category": category
        }

    else:
        entry = {
            "question": "how can i contact customer support",
            "answer": "You can contact customer support through the Help section.",
            "keywords": "support help contact",
            "category": category
        }

    personalized_entries.append(entry)

# Combine all entries
all_entries = fixed_entries + personalized_entries

df = pd.DataFrame(all_entries)
print(df)




                                      question  \
0                       what is the annual fee   
1                        how to reset password   
2                  what are your working hours   
3                        how can i pay the fee   
4  how do i update my registered mobile number   
5              how can i get a payment receipt   

                                              answer                 keywords  \
0                          The annual fee is Rs 500.    fee cost price charge   
1                   Go to Settings > Reset Password.     password reset login   
2                          We are open 9 AM to 5 PM.   hours timing open time   
3         You can pay via UPI, card, or net banking.      pay payment upi fee   
4  Go to Settings > Account > Mobile Number to up...     mobile number update   
5  You can download your payment receipt from the...  receipt payment billing   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  account  
5  

In [ ]:
# Q2. Generate and Score a Hypothesis
def score_query(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        # Combine question and keywords
        text = (
            row["question"].lower()
            + " "
            + row["keywords"].lower()
        )

        text_words = set(text.split())

        # Count matching words
        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    # Sort by confidence, highest first
    results.sort(key=lambda x: x["confidence"], reverse=True)

    return results


query = input("Enter your query: ")
results = score_query(query, df)

if results:
    for result in results:
        print(
            f"\nQuestion: {result['question']}"
            f"\nAnswer: {result['answer']}"
            f"\nCategory: {result['category']}"
            f"\nConfidence: {result['confidence']}"
        )
else:
    print("No matching FAQ found.")




Enter your query: what is the annual fee

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Confidence: 5

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Confidence: 2

Question: what are your working hours
Answer: We are open 9 AM to 5 PM.
Category: general
Confidence: 1


In [ ]:
# Q3. Find Questions From Same Category
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]


# Category of first personalized entry
personalized_category = personalized_entries[0]["category"]
print("Category selected:", personalized_category)
category_result = same_category(personalized_category, df)
print(category_result[["question", "category"]])

Category selected: account
                                      question category
1                        how to reset password  account
4  how do i update my registered mobile number  account


In [ ]:
# Q4. Add New Keyword and Save DataFrame to CSV

# Pick the first FAQ entry
entry_index = 0

print("Selected question:")
print(df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

if new_keyword:
    df.loc[entry_index, "keywords"] += " " + new_keyword

print("\nUpdated Entry:")
print(df.loc[entry_index])

# Save complete DataFrame
csv_filename = roll_number + "_faq_data.csv"
df.to_csv(csv_filename, index=False)
print("\nUpdated DataFrame saved as:", csv_filename)

Selected question:
what is the annual fee
Enter a new keyword to add: fees

Updated Entry:
question             what is the annual fee
answer            The annual fee is Rs 500.
keywords    fee cost price charge fees fees
category                            billing
Name: 0, dtype: object

Updated DataFrame saved as: 1024160013_faq_data.csv


In [ ]:
# Q5. Count FAQ Entries Per Category
category_counts = df.groupby("category").size()
print(category_counts)

category
account    2
billing    3
general    1
dtype: int64


In [ ]:
# Q6. Tie Handling
def score_query_with_tie(query, df):

    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        text = (
            row["question"].lower()
            + " "
            + row["keywords"].lower()
        )

        text_words = set(text.split())

        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    # No matches
    if not results:
        print("No matching FAQ found.")
        return

    # Sort by confidence
    results.sort(
        key=lambda x: x["confidence"],
        reverse=True
    )

    highest_score = results[0]["confidence"]

    # Get all entries having highest score
    top_matches = [
        result
        for result in results
        if result["confidence"] == highest_score
    ]

    print("\nHighest Confidence:", highest_score)

    if len(top_matches) > 1:

        print("\nTIE DETECTED!")
        print("Multiple FAQs have the same highest confidence.")

        for result in top_matches:
            print(
                f"\nQuestion: {result['question']}"
                f"\nAnswer: {result['answer']}"
                f"\nCategory: {result['category']}"
                f"\nConfidence: {result['confidence']}"
            )

    else:

        print("\nSingle best match:")

        result = top_matches[0]

        print(
            f"\nQuestion: {result['question']}"
            f"\nAnswer: {result['answer']}"
            f"\nCategory: {result['category']}"
            f"\nConfidence: {result['confidence']}"
        )



# Q6 Demonstration 1: Query producing a tie
print("\n========== Q6: TIE QUERY ==========")
score_query_with_tie("fee", df)



# Q6 Demonstration 2: Query without a tie
print("\n========== Q6: NON-TIE QUERY ==========")
score_query_with_tie("password", df)


========== Q6: TIE QUERY ==========

Highest Confidence: 1

TIE DETECTED!
Multiple FAQs have the same highest confidence.

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Confidence: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Confidence: 1

========== Q6: NON-TIE QUERY ==========

Highest Confidence: 1

Single best match:

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Confidence: 1
